# 🛒 Práctica 2 · RA2: Decision Tree y Random Forest
## Predicción de intención de compra en comercio electrónico con datos reales

* **Módulo:** M3 · Programació d'Intel·ligència Artificial (RA2 · Implementación y evaluación de modelos de IA)
* **Dataset:** *Online Shoppers Purchasing Intention* (UCI Machine Learning Repository · Sakar y Kastro, DOI `10.24432/C5F88Q`)
* **Entorno:** Google Colab / Jupyter Notebook (100% reproducible)

---

### 📋 Índice del Notebook
1. **Contexto del problema y carga de datos**
2. **Exploración inicial (EDA)** y análisis del desequilibrio de clases
3. **Preparación de los datos** (preprocesamiento reproducible, sin *data leakage*)
4. **Decision Tree**: entrenamiento, visualización e interpretación de reglas reales
5. **Control de complejidad y *overfitting*** (análisis de hiperparámetros)
6. **Regiones de decisión** en 2D y su correspondencia geométrica con el árbol
7. **Random Forest**: entrenamiento y justificación de hiperparámetros
8. **Evaluación de modelos**: métricas y matrices de confusión comparadas
9. **Importancia de variables** (*Feature Importance*) y análisis de causalidad
10. **Comparación final**: Decision Tree vs. Random Forest
11. **Análisis de errores**: estudio detallado de casos reales mal clasificados
12. **Conclusiones finales**


---
## 1. Contexto y carga del dataset

En esta práctica trabajamos con el dataset público **Online Shoppers Purchasing Intention** del repositorio UCI Machine Learning. El objetivo de negocio consiste en **predecir si una sesión de navegación en una tienda online finalizará en compra (`Revenue = True`) o no (`Revenue = False`)** a partir de métricas de comportamiento durante la sesión y metadatos del usuario.

### Descripción de las 18 variables del dataset

| Categoría | Variable | Tipo | Descripción |
| :--- | :--- | :--- | :--- |
| **Navegación (Páginas y duración)** | `Administrative` | Numérica (int) | Número de páginas de gestión de cuenta visitadas en la sesión. |
| | `Administrative_Duration` | Numérica (float) | Tiempo total (segundos) dedicado a páginas administrativas. |
| | `Informational` | Numérica (int) | Número de páginas informativas visitadas (contacto, FAQ, empresa). |
| | `Informational_Duration` | Numérica (float) | Tiempo total (segundos) en páginas informativas. |
| | `ProductRelated` | Numérica (int) | Número de páginas de catálogo/producto visitadas. |
| | `ProductRelated_Duration` | Numérica (float) | Tiempo total (segundos) en páginas de producto. |
| **Métricas Google Analytics** | `BounceRates` | Numérica (float) | Tasa media de rebote de las páginas visitadas (usuarios que entran y salen sin interactuar). |
| | `ExitRates` | Numérica (float) | Tasa media de abandono de las páginas visitadas en la sesión. |
| | `PageValues` | Numérica (float) | Valor económico medio aportado por las páginas visitadas antes de completar una transacción previa. |
| **Contexto temporal** | `SpecialDay` | Numérica (float) | Proximidad de la visita a una fecha especial (San Valentín, Día de la Madre, etc.), entre `0.0` y `1.0`. |
| | `Month` | Categórica (str) | Mes del año en que se realizó la sesión (`Feb`, `Mar`, `May`, ..., `Dec`). |
| | `Weekend` | Booleana (bool) | Indica si la sesión ocurrió en fin de semana (`True` / `False`). |
| ** Entorno y usuario** | `OperatingSystems` | Categórica codificada (int) | Identificador anónimo del sistema operativo (1 a 8). |
| | `Browser` | Categórica codificada (int) | Identificador anónimo del navegador web (1 a 13). |
| | `Region` | Categórica codificada (int) | Región geográfica desde la que accede el usuario (1 a 9). |
| | `TrafficType` | Categórica codificada (int) | Canal de origen del tráfico (directo, SEO, SEM, redes sociales, etc., 1 a 20). |
| | `VisitorType` | Categórica (str) | Tipo de visitante: `Returning_Visitor`, `New_Visitor` u `Other`. |
| **Variable Objetivo** | **`Revenue`** | **Booleana (bool)** | **Indica si la sesión terminó en compra (`True`, ~15,5%) o no (`False`, ~84,5%).** |

> 💡 **Nota de reproducibilidad para Google Colab:** La siguiente celda comprueba si el archivo `online_shoppers_intention.csv` está disponible localmente y, en caso contrario, lo descarga automáticamente del repositorio oficial de UCI Machine Learning.


In [1]:
# Importación de librerías principales
import os
import urllib.request
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

# Configuración estética global de gráficos y tablas
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Carga reproducible del dataset (local o descarga automática en Google Colab)
CSV_FILENAME = "online_shoppers_intention.csv"
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00468/online_shoppers_intention.csv"

if not os.path.exists(CSV_FILENAME):
    print(f"Descargando '{CSV_FILENAME}' desde UCI Machine Learning Repository...")
    urllib.request.urlretrieve(UCI_URL, CSV_FILENAME)
    print("Descarga completada con éxito.")
else:
    print(f"Archivo '{CSV_FILENAME}' encontrado localmente.")

df = pd.read_csv(CSV_FILENAME)
print(f"Dataset cargado correctamente: {df.shape[0]:,} filas y {df.shape[1]} columnas.")
df.head()


Archivo 'online_shoppers_intention.csv' encontrado localmente.
Dataset cargado correctamente: 12,330 filas y 18 columnas.


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0000,0,0.0000,1,0.0000,0.2000,0.2000,0.0000,0.0000,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0000,0,0.0000,2,64.0000,0.0000,0.1000,0.0000,0.0000,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0000,0,0.0000,1,0.0000,0.2000,0.2000,0.0000,0.0000,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0000,0,0.0000,2,2.6667,0.0500,0.1400,0.0000,0.0000,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0000,0,0.0000,10,627.5000,0.0200,0.0500,0.0000,0.0000,Feb,3,3,1,4,Returning_Visitor,True,False


---
## 2. Exploración inicial (EDA)

Antes de construir cualquier modelo, realizamos un análisis exploratorio exhaustivo para comprender la estructura de los datos, la distribución de las variables y los patrones que diferencian las sesiones con compra de las sesiones sin compra.

### 2.1. Dimensiones, tipos de datos y comprobación de valores nulos


In [2]:
# Dimensiones, tipos de datos y valores nulos
resumen_estructura = pd.DataFrame({
    'Tipo de dato': df.dtypes,
    'Valores No Nulos': df.notnull().sum(),
    'Valores Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df)) * 100,
    'Valores Únicos': df.nunique()
})

print(f"Dimensiones del dataset: {df.shape[0]} sesiones x {df.shape[1]} columnas")
print(f"Total de valores nulos en todo el dataset: {df.isnull().sum().sum()}")
print(f"Filas exactamente duplicadas: {df.duplicated().sum()} ({df.duplicated().mean()*100:.2f}%)")
display(resumen_estructura)


Dimensiones del dataset: 12330 sesiones x 18 columnas
Total de valores nulos en todo el dataset: 0
Filas exactamente duplicadas: 125 (1.01%)


,Tipo de dato,Valores No Nulos,Valores Nulos,% Nulos,Valores Únicos
Administrative,int64,12330,0,0.0000,27
Administrative_Duration,float64,12330,0,0.0000,3335
Informational,int64,12330,0,0.0000,17
Informational_Duration,float64,12330,0,0.0000,1258
ProductRelated,int64,12330,0,0.0000,311
ProductRelated_Duration,float64,12330,0,0.0000,9551
BounceRates,float64,12330,0,0.0000,1872
ExitRates,float64,12330,0,0.0000,4777
PageValues,float64,12330,0,0.0000,2704
SpecialDay,float64,12330,0,0.0000,6


**Observaciones iniciales:**
* El dataset cuenta exactamente con **12.330 filas** (sesiones independientes) y **18 columnas**.
* **No existe ningún valor nulo (`0` nulos)** en ninguna de las columnas, por lo que no se requiere imputación de datos faltantes.
* Existen 125 filas duplicadas (principalmente sesiones muy cortas donde el usuario entra en una sola página de producto con duración 0 y rebota inmediatamente). Las conservamos porque representan un comportamiento real y frecuente en comercio electrónico (rebotes instantáneos).
* En cuanto a los tipos:
  * **10 variables numéricas continuas/discretas** (`Administrative`, `Administrative_Duration`, `Informational`, `Informational_Duration`, `ProductRelated`, `ProductRelated_Duration`, `BounceRates`, `ExitRates`, `PageValues`, `SpecialDay`).
  * **4 variables enteras que representan códigos de categorías nominales** (`OperatingSystems`, `Browser`, `Region`, `TrafficType`).
  * **2 variables categóricas en texto (`str`/`object`)**: `Month` (10 meses presentes; no hay datos de enero ni abril) y `VisitorType` (3 categorías).
  * **2 variables booleanas (`bool`)**: `Weekend` (predictora) y `Revenue` (objetivo).

---
### 2.2. Distribución de la variable objetivo (`Revenue`)


In [3]:
# Distribución de la variable objetivo Revenue
conteo_revenue = df['Revenue'].value_counts()
porcentaje_revenue = df['Revenue'].value_counts(normalize=True) * 100

tabla_revenue = pd.DataFrame({
    'Sesiones (Frecuencia absoluta)': conteo_revenue,
    'Porcentaje (%)': porcentaje_revenue
})
tabla_revenue.index = ['No Compra (False)', 'Compra (True)']
display(tabla_revenue)

# Visualización gráfica del desequilibrio de clases
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

colores = ['#4C72B0', '#55A868']
barras = axes[0].bar(['No Compra (False)', 'Compra (True)'], conteo_revenue.values, color=colores, edgecolor='black', width=0.55)
axes[0].set_title('Frecuencia absoluta de sesiones por Revenue', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Número de sesiones')
for bar, pct in zip(barras, porcentaje_revenue.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 180,
                 f"{bar.get_height():,}\n({pct:.2f}%)", ha='center', va='bottom', fontweight='bold')
axes[0].set_ylim(0, 12000)

axes[1].pie(conteo_revenue.values, labels=['No Compra (False)', 'Compra (True)'],
            autopct='%1.2f%%', startangle=140, colors=colores, explode=(0, 0.08),
            wedgeprops={'edgecolor': 'black'})
axes[1].set_title('Proporción de la variable objetivo Revenue', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


,Sesiones (Frecuencia absoluta),Porcentaje (%)
No Compra (False),10422,84.5255
Compra (True),1908,15.4745


---
### 2.3. Distribución y valores extremos (*outliers*) de las variables numéricas

Analizamos los estadísticos descriptivos (media, desviación típica, cuartiles y máximo) de las variables numéricas y representamos sus distribuciones e intervalos intercuartílicos mediante histogramas y *boxplots*.


In [4]:
# Estadísticos descriptivos de las variables numéricas
num_features = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'
]

desc_num = df[num_features].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).T
desc_num['Asimetría (Skewness)'] = df[num_features].skew()
display(desc_num)


,count,mean,std,min,25%,50%,75%,95%,99%,max,Asimetría (Skewness)
Administrative,12330.0000,2.3152,3.3218,0.0000,0.0000,1.0000,4.0000,9.0000,14.0000,27.0000,1.9604
Administrative_Duration,12330.0000,80.8186,176.7791,0.0000,0.0000,7.5000,93.2562,348.2664,830.5870,3398.7500,5.6157
Informational,12330.0000,0.5036,1.2702,0.0000,0.0000,0.0000,0.0000,3.0000,6.0000,24.0000,4.0365
Informational_Duration,12330.0000,34.4724,140.7493,0.0000,0.0000,0.0000,0.0000,195.0000,716.3900,2549.3750,7.5792
ProductRelated,12330.0000,31.7315,44.4755,0.0000,7.0000,18.0000,38.0000,109.0000,221.0000,705.0000,4.3415
ProductRelated_Duration,12330.0000,1194.7462,1913.6693,0.0000,184.1375,598.9369,1464.1572,4300.2891,8701.1427,63973.5222,7.2632
BounceRates,12330.0000,0.0222,0.0485,0.0000,0.0000,0.0031,0.0168,0.2000,0.2000,0.2000,2.9479
ExitRates,12330.0000,0.0431,0.0486,0.0000,0.0143,0.0252,0.0500,0.2000,0.2000,0.2000,2.1488
PageValues,12330.0000,5.8893,18.5684,0.0000,0.0000,0.0000,0.0000,38.1605,85.4985,361.7637,6.3830
SpecialDay,12330.0000,0.0614,0.1989,0.0000,0.0000,0.0000,0.0000,0.6000,1.0000,1.0000,3.3027


In [5]:
# Histogramas y Boxplots para detectar asimetría y valores extremos (outliers)
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, col in enumerate(num_features):
    sns.boxplot(x=df[col], ax=axes[i], color='#64B5CD', fliersize=2)
    axes[i].set_title(f'Boxplot: {col}', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Distribución y valores extremos (Outliers) en variables numéricas', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Análisis de las variables numéricas y valores extremos:**
1. **Fuerte asimetría positiva (cola larga a la derecha):** Todas las variables de conteo de páginas, duración y `PageValues` presentan una asimetría positiva muy pronunciada (`Skewness > 3`). La gran mayoría de las sesiones visitan pocas páginas y duran pocos minutos, pero existen sesiones extremas (*heavy shoppers* o pestañas dejadas abiertas) con hasta `705` páginas de producto visitadas o duraciones superiores a `63.973` segundos (~17 horas).
2. **Acumulación en cero (*zero-inflation*):**
   * En `Informational` e `Informational_Duration`, más del 75% de las sesiones tienen valor `0` (el tercer cuartil `75%` es `0.0`).
   * En `PageValues`, al menos el 75% de las sesiones tienen valor `0.0`, pero el percentil 95 sube a `38.15` y el máximo alcanza `361.76`.
3. **Tasas acotadas (`BounceRates`, `ExitRates`, `SpecialDay`):** Están acotadas en el rango `[0.0, 0.2]` para las tasas de rebote/salida y `[0.0, 1.0]` para `SpecialDay`.
4. **¿Debemos eliminar estos *outliers*?** En árboles de decisión y Random Forest **no es necesario ni recomendable eliminarlos**, ya que una alta duración en páginas de producto o un `PageValues` muy elevado son precisamente señales legítimas de usuarios con altísima intención de compra, y los árboles realizan particiones por umbral que no se ven distorsionadas por valores extremos.

---
### 2.4. Frecuencias de las variables categóricas y tasa de conversión por categoría


In [6]:
# Análisis de frecuencias y tasa de compra en variables categóricas y booleanas
cat_explore = ['Month', 'VisitorType', 'Weekend', 'TrafficType', 'Region', 'Browser', 'OperatingSystems']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Frecuencia por Mes y tasa de conversión
orden_meses = ['Feb', 'Mar', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
sns.countplot(data=df, x='Month', order=orden_meses, hue='Revenue', palette=colores, ax=axes[0, 0])
axes[0, 0].set_title('Sesiones por Mes desglosadas por Compra (Revenue)', fontweight='bold')
axes[0, 0].set_ylabel('Número de sesiones')

# 2. Tasa de compra (%) por Mes
tasa_mes = df.groupby('Month')['Revenue'].mean().reindex(orden_meses) * 100
sns.barplot(x=tasa_mes.index, y=tasa_mes.values, color='#DD8452', edgecolor='black', ax=axes[0, 1])
axes[0, 1].axhline(df['Revenue'].mean()*100, color='red', linestyle='--', label=f'Media global ({df["Revenue"].mean()*100:.1f}%)')
axes[0, 1].set_title('Tasa de conversión (% Compra) por Mes', fontweight='bold')
axes[0, 1].set_ylabel('% de sesiones con compra')
axes[0, 1].legend()

# 3. Frecuencia y tasa por Tipo de Visitante
tasa_visitor = df.groupby('VisitorType')['Revenue'].agg(['count', 'mean'])
tasa_visitor['mean'] *= 100
sns.countplot(data=df, x='VisitorType', hue='Revenue', palette=colores, ax=axes[1, 0])
axes[1, 0].set_title('Sesiones por Tipo de Visitante (VisitorType)', fontweight='bold')
axes[1, 0].set_ylabel('Número de sesiones')

# 4. Fin de semana vs Día laborable
sns.countplot(data=df, x='Weekend', hue='Revenue', palette=colores, ax=axes[1, 1])
axes[1, 1].set_title('Sesiones en Fin de Semana (Weekend) vs Laborable', fontweight='bold')
axes[1, 1].set_ylabel('Número de sesiones')

plt.tight_layout()
plt.show()

print("Resumen de VisitorType (Frecuencia y % Compra):")
display(tasa_visitor.rename(columns={'count': 'Total Sesiones', 'mean': '% Conversión (Compra)'}))


Resumen de VisitorType (Frecuencia y % Compra):


,Total Sesiones,% Conversión (Compra)
VisitorType,,
New_Visitor,1694,24.9115
Other,85,18.8235
Returning_Visitor,10551,13.9323


**Hallazgos sobre las variables categóricas:**
* **Estacionalidad marcada (`Month`):** Los meses con mayor volumen de tráfico son **mayo (3.364)**, **noviembre (2.998)**, **marzo (1.907)** y **diciembre (1.727)**. Sin embargo, la **tasa de conversión se dispara en noviembre (~25,35%, campaña de *Black Friday* / pre-Navidad)** y octubre (~20,95%), mientras que en febrero (2,17%) y mayo (10,85%) es mucho menor.
* **Comportamiento según `VisitorType`:** Aunque el **85,57% de las sesiones corresponden a visitantes recurrentes (`Returning_Visitor`, 10.551 sesiones)** con una tasa de conversión del `13,93%`, los **nuevos visitantes (`New_Visitor`, 1.694 sesiones) convierten casi el doble (`24,91%`)**. Esto ocurre porque muchos usuarios recurrentes entran a comparar precios o hacer seguimiento, mientras que quienes llegan por primera vez desde enlaces directos de producto suelen tener una intención de compra más inmediata.
* **Fin de semana (`Weekend`):** El 23,26% de las sesiones ocurre en fin de semana, con una tasa de conversión ligeramente superior (`17,40%` en fin de semana frente a `14,89%` entre semana).

---
### 2.5. Relaciones entre duración, páginas visitadas, tasas de rebote y compra (Diferencias entre sesiones con y sin compra)


In [7]:
# Comparativa de medias y medianas entre sesiones CON compra (True) y SIN compra (False)
comparativa_grupos = df.groupby('Revenue')[num_features].agg(['mean', 'median']).T
comparativa_grupos.columns = ['Sin Compra (False)', 'Con Compra (True)']
display(comparativa_grupos)

# Matriz de correlación entre variables numéricas y Revenue
plt.figure(figsize=(11, 8))
corr_matrix = df[num_features + ['Weekend', 'Revenue']].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-0.3, vmax=1.0, linewidths=0.5)
plt.title('Matriz de Correlación de Pearson (Variables Numéricas y Revenue)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


Sin Compra (False)  Con Compra (True)
Administrative          mean                2.1177             3.3936
                        median              0.0000             2.0000
Administrative_Duration mean               73.7401           119.4832
                        median              0.0000            52.3667
Informational           mean                0.4518             0.7862
                        median              0.0000             0.0000
Informational_Duration  mean               30.2362            57.6114
                        median              0.0000             0.0000
ProductRelated          mean               28.7146            48.2102
                        median             16.0000            29.0000
ProductRelated_Duration mean             1069.9878          1876.2096
                        median            510.1900          1109.9062
BounceRates             mean                0.0253             0.0051
                        median              0.0043             0.0000
ExitRates               mean                0.0474             0.0196
                        median              0.0286             0.0160
PageValues              mean                1.9760            27.2645
                        median              0.0000            16.7581
SpecialDay              mean                0.0684             0.0232
                        median              0.0000             0.0000

In [8]:
# Visualización de las diferencias clave entre sesiones con y sin compra
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# 1. PageValues según Revenue (escala log1p para visualizar mejor la distribución)
sns.boxplot(data=df, x='Revenue', y=np.log1p(df['PageValues']), palette=colores, ax=axes[0])
axes[0].set_title('log(1 + PageValues) según Compra', fontweight='bold')
axes[0].set_ylabel('log(1 + PageValues)')

# 2. ExitRates según Revenue
sns.boxplot(data=df, x='Revenue', y='ExitRates', palette=colores, ax=axes[1])
axes[1].set_title('Tasa de Salida (ExitRates) según Compra', fontweight='bold')

# 3. Relación entre Páginas de Producto y Duración según Revenue
sns.scatterplot(
    data=df.sample(2500, random_state=RANDOM_STATE),
    x='ProductRelated', y='ProductRelated_Duration',
    hue='Revenue', palette=colores, alpha=0.65, ax=axes[2]
)
axes[2].set_title('Páginas de Producto vs Duración (Muestra n=2500)', fontweight='bold')
axes[2].set_xlim(0, 250)
axes[2].set_ylim(0, 15000)

plt.tight_layout()
plt.show()


**Diferencias clave entre sesiones con compra (`Revenue = True`) y sin compra (`Revenue = False`):**
1. **`PageValues` es el diferenciador más potente:** La media de `PageValues` en las sesiones con compra es de **`27.26`** (mediana `16.76`), mientras que en las sesiones sin compra es de apenas **`1.98`** (mediana `0.00`). Tiene una correlación lineal de `+0.49` con `Revenue`.
2. **Mayor navegación y tiempo en páginas de producto:** Quienes compran visitan de media **`48.21` páginas de producto** dedicando **`1.876` segundos (~31 min)**, frente a **`28.71` páginas** y **`1.070` segundos (~18 min)** en quienes no compran. Además, existe una alta colinealidad (`r = 0.86`) entre `ProductRelated` y `ProductRelated_Duration`.
3. **Menores tasas de rebote y salida (`BounceRates` y `ExitRates`):** Las sesiones que terminan en compra tienen una tasa media de rebote 5 veces menor (`0.0051` vs `0.0253`) y una tasa de salida más de 2 veces menor (`0.0196` vs `0.0474`).

---
### ❓ Pregunta de análisis (Sección 2)
> **¿Por qué una *accuracy* alta puede resultar engañosa si solo el 15,5% de las sesiones termina en compra?**

**Respuesta razonada:**
En un dataset con desequilibrio de clases donde el **84,53% de las sesiones no terminan en compra (`False`)** y solo el **15,47% terminan en compra (`True`)**, un clasificador completamente trivial o "ingenuo" (*Dummy Classifier*) que prediga **siempre `No Compra (0)` para absolutamente todos los usuarios** obtendría automáticamente una **Accuracy del 84,53%**:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} = \frac{0 + 10.422}{12.330} = 84,53\%$$

A pesar de parecer una cifra alta (>84%), ese modelo sería **totalmente inútil para el negocio**, ya que tendría un **Recall (sensibilidad) del 0%** sobre la clase minoritaria (`Compra = True`), siendo incapaz de detectar ni un solo comprador potencial. Por ello, es imprescindible evaluar los modelos mediante métricas centradas en la clase positiva minoritaria:
* **Precision (Precisión):** De las sesiones que el modelo predice como compra, ¿qué porcentaje compra realmente?
* **Recall (Sensibilidad / Exhaustividad):** De todas las sesiones que realmente terminaron en compra, ¿qué porcentaje logra detectar el modelo?
* **F1-Score:** Media armónica entre *Precision* y *Recall* ($F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$), que penaliza fuertemente si cualquiera de las dos es baja.


---
## 3. Preparación de los datos

Seguimos un flujo de preprocesamiento riguroso y reproducible en 6 pasos:
1. **Separación de variable objetivo (`y`) y variables predictoras (`X`):** `Revenue` se transforma a entero binario (`0` = No Compra, `1` = Compra).
2. **Identificación de tipos de variables:**
   * **Numéricas continuas y discretas (10):** `Administrative`, `Administrative_Duration`, `Informational`, `Informational_Duration`, `ProductRelated`, `ProductRelated_Duration`, `BounceRates`, `ExitRates`, `PageValues`, `SpecialDay`.
   * **Categóricas nominales ya codificadas como enteros (4):** `OperatingSystems`, `Browser`, `Region`, `TrafficType`.
   * **Booleanas (1):** `Weekend` (se convierte a entero binario `0`/`1`).
   * **Categóricas en formato texto (2):** `Month` y `VisitorType`.
3. **Codificación de `Month` y `VisitorType`:** Aplicamos `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` sobre `Month` y `VisitorType` para crear variables indicadoras binarias sin imponer un orden numérico artificial entre tipos de visitante ni perder el efecto estacional específico de meses clave como noviembre (`Month_Nov`) o mayo (`Month_May`).
4. **Partición estratificada en entrenamiento (80%) y test (20%):** Utilizamos `train_test_split(..., stratify=y, random_state=42)` para garantizar que tanto entrenamiento como test conserven exactamente el `15,47%` de compras.
5. **Fijación de semilla (`random_state = 42`):** Garantiza resultados 100% reproducibles en cualquier ejecución.
6. **Ajuste de transformaciones únicamente sobre entrenamiento (`fit_transform` en train, `transform` en test):** Evita cualquier fuga de información (*data leakage*).


In [9]:
# 1. Separar Revenue como variable objetivo (y) y el resto como predictoras (X)
X = df.drop(columns=['Revenue']).copy()
y = df['Revenue'].astype(int)

# 2. Identificar variables numéricas, booleanas y categóricas
num_cols = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay',
    'OperatingSystems', 'Browser', 'Region', 'TrafficType'
]
bool_cols = ['Weekend']
cat_cols = ['Month', 'VisitorType']

# Convertimos la columna booleana Weekend a 0/1
X['Weekend'] = X['Weekend'].astype(int)

# 4 y 5. División en entrenamiento (80%) y test (20%) mediante partición estratificada y random_state fijo
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

# 3 y 6. Codificación con ColumnTransformer ajustado ÚNICAMENTE sobre X_train
preprocessor = ColumnTransformer(
    transformers=[
        ('num_bool', 'passthrough', num_cols + bool_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ],
    verbose_feature_names_out=False
)

# Ajustamos SOLO con X_train y transformamos ambos conjuntos
X_train_proc = pd.DataFrame(
    preprocessor.fit_transform(X_train),
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)
X_test_proc = pd.DataFrame(
    preprocessor.transform(X_test),
    columns=preprocessor.get_feature_names_out(),
    index=X_test.index
)

print(f"Dimensiones X_train procesado: {X_train_proc.shape} | Proporción Revenue=1 en train: {y_train.mean()*100:.2f}%")
print(f"Dimensiones X_test procesado:  {X_test_proc.shape}  | Proporción Revenue=1 en test:  {y_test.mean()*100:.2f}%")
print(f"\nTotal de variables tras One-Hot Encoding ({len(X_train_proc.columns)}):")
print(list(X_train_proc.columns))


Dimensiones X_train procesado: (9864, 28) | Proporción Revenue=1 en train: 15.47%
Dimensiones X_test procesado:  (2466, 28)  | Proporción Revenue=1 en test:  15.49%

Total de variables tras One-Hot Encoding (28):
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Weekend', 'Month_Aug', 'Month_Dec', 'Month_Feb', 'Month_Jul', 'Month_June', 'Month_Mar', 'Month_May', 'Month_Nov', 'Month_Oct', 'Month_Sep', 'VisitorType_New_Visitor', 'VisitorType_Other', 'VisitorType_Returning_Visitor']


---
### 🧠 Justificación técnica y preguntas de análisis (Sección 3)

#### 1. ¿Por qué Decision Tree y Random Forest no necesitan escalado (normalización o estandarización) para realizar sus divisiones?
A diferencia de algoritmos basados en distancias euclídeas (como *K-Nearest Neighbors*, *K-Means* o *SVM*) o en descenso por gradiente (como la Regresión Logística o Redes Neuronales), **los árboles de decisión son invariantes frente a transformaciones monótonas crecientes de las variables**:
* Para dividir un nodo utilizando una variable continua $X_j$, el algoritmo CART ordena los valores únicos de $X_j$ y busca el umbral de corte $t$ ($X_j \le t$ frente a $X_j > t$) que maximiza la reducción de impureza de Gini (o entropía).
* La impureza de Gini depende **únicamente de las proporciones de clases a cada lado del corte**, no de la magnitud numérica de $X_j$.
* Si estandarizamos $X_j$ mediante $Z_j = \frac{X_j - \mu_j}{\sigma_j}$ (con $\sigma_j > 0$), el orden de las muestras no cambia en absoluto: el umbral óptimo $t$ simplemente se transforma en $t' = \frac{t - \mu_j}{\sigma_j}$, dejando exactamente las mismas muestras a la izquierda y a la derecha del nodo. Además, mantener las variables en sus unidades originales (segundos, número de páginas, porcentajes) facilita enormemente la interpretación humana de las reglas del árbol.

#### 2. Desequilibrio de clases y uso de `class_weight`
Dado que la clase positiva (`Compra = 1`) representa solo el `15,47%` de los datos, al entrenar un árbol con pesos uniformes (`class_weight=None`), el criterio de Gini prioriza reducir errores en la clase mayoritaria (`No Compra = 0`), lo que suele producir una **Precision alta pero un Recall moderado/bajo** (se escapan compradores reales).
* Si utilizamos `class_weight='balanced'`, el algoritmo pondera cada clase de forma inversamente proporcional a su frecuencia ($w_c = \frac{N}{2 \cdot N_c}$: peso `~0.59` para clase 0 y `~3.23` para clase 1). Esto **incrementa drásticamente el Recall** (detectamos más compradores) a cambio de reducir la Precision (aumentan los falsos positivos).
* En esta práctica **analizaremos y compararemos ambas variantes** (sin balanceo y con `class_weight='balanced'`), utilizando el **F1-score** y el **ROC-AUC** como métricas principales de decisión.

#### 3. ❓ Pregunta de análisis: ¿Qué fuga de información (*data leakage*) se produciría si ajustáramos la codificación o cualquier transformación utilizando también el conjunto de test?
Si ejecutásemos `.fit()` del preprocesador sobre el dataset completo `df` **antes** de dividir en entrenamiento y test, estaríamos cometiendo **fuga de información (*data leakage*)**:
1. **En la codificación categórica (`OneHotEncoder` / `TargetEncoder`):** El modelo conocería de antemano categorías que quizá solo existen en el conjunto de test (o frecuencias/tasas de compra del test si se usara codificación por frecuencia o *target encoding*).
2. **En transformaciones estadísticas (imputación por media/mediana, escalado, selección de variables):** Los estadísticos calculados incorporarían información de la distribución del conjunto de test, haciendo que el modelo se entrene con información del "futuro".
3. **Consecuencia:** Las métricas obtenidas al evaluar sobre `X_test` serían **artificialmente optimistas** y engañosas, y cuando el modelo se desplegara en producción frente a sesiones verdaderamente nuevas, su rendimiento real caería.


---
## 4. Decision Tree (Árbol de Decisión)

En esta sección entrenamos primero un **árbol de decisión base sin restricciones (`max_depth=None`)** para observar su comportamiento natural, y a continuación un **árbol de decisión podado e interpretable (`max_depth=3`)** para visualizar su estructura completa y extraer reglas reales de negocio.


In [10]:
# Función auxiliar reutilizable para evaluar modelos y devolver un diccionario de métricas
def evaluar_modelo(nombre, modelo, X_tr, y_tr, X_te, y_te, mostrar=True):
    y_pred_tr = modelo.predict(X_tr)
    y_pred_te = modelo.predict(X_te)
    y_prob_te = modelo.predict_proba(X_te)[:, 1]
    
    metricas = {
        'Modelo': nombre,
        'Acc Train': accuracy_score(y_tr, y_pred_tr),
        'F1 Train': f1_score(y_tr, y_pred_tr),
        'Accuracy (Test)': accuracy_score(y_te, y_pred_te),
        'Precision (Test)': precision_score(y_te, y_pred_te),
        'Recall (Test)': recall_score(y_te, y_pred_te),
        'F1-Score (Test)': f1_score(y_te, y_pred_te),
        'ROC-AUC (Test)': roc_auc_score(y_te, y_prob_te)
    }
    if mostrar:
        print(f"📊 Resultados de: {nombre}")
        print(f"   • Entrenamiento -> Accuracy: {metricas['Acc Train']:.4f} | F1-Score: {metricas['F1 Train']:.4f}")
        print(f"   • Test          -> Accuracy: {metricas['Accuracy (Test)']:.4f} | Precision: {metricas['Precision (Test)']:.4f} | Recall: {metricas['Recall (Test)']:.4f} | F1-Score: {metricas['F1-Score (Test)']:.4f} | ROC-AUC: {metricas['ROC-AUC (Test)']:.4f}\n")
    return metricas

# 1. Árbol de decisión inicial sin límite de profundidad
dt_sin_limite = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_sin_limite.fit(X_train_proc, y_train)

res_dt_sin_limite = evaluar_modelo("Decision Tree (Sin límite, max_depth=None)", dt_sin_limite, X_train_proc, y_train, X_test_proc, y_test)
print(f"   Profundidad alcanzada: {dt_sin_limite.get_depth()} niveles | Número total de hojas: {dt_sin_limite.get_n_leaves()}")

# 2. Árbol de decisión de profundidad controlada (max_depth=3) para visualización e interpretación directa
dt_interpretable = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
dt_interpretable.fit(X_train_proc, y_train)

res_dt_interpretable = evaluar_modelo("Decision Tree Interpretable (max_depth=3)", dt_interpretable, X_train_proc, y_train, X_test_proc, y_test)


📊 Resultados de: Decision Tree (Sin límite, max_depth=None)
   • Entrenamiento -> Accuracy: 1.0000 | F1-Score: 1.0000
   • Test          -> Accuracy: 0.8564 | Precision: 0.5355 | Recall: 0.5524 | F1-Score: 0.5438 | ROC-AUC: 0.7323

   Profundidad alcanzada: 24 niveles | Número total de hojas: 815
📊 Resultados de: Decision Tree Interpretable (max_depth=3)
   • Entrenamiento -> Accuracy: 0.8994 | F1-Score: 0.6565
   • Test          -> Accuracy: 0.8970 | Precision: 0.6988 | Recall: 0.5890 | F1-Score: 0.6392 | ROC-AUC: 0.9011



### 4.1. Visualización gráfica y textual del Árbol de Decisión

Visualizamos el árbol de profundidad 3 (que coincide exactamente con los 3 primeros niveles del árbol completo, pero evita las 815 hojas ilegibles del árbol sobreajustado) tanto de forma gráfica (`plot_tree`) como en texto estructurado (`export_text`).


In [11]:
# Visualización gráfica del árbol de decisión (primeros 3 niveles)
plt.figure(figsize=(22, 10))
plot_tree(
    dt_interpretable,
    feature_names=list(X_train_proc.columns),
    class_names=['No Compra (0)', 'Compra (1)'],
    filled=True,
    rounded=True,
    proportion=False,
    precision=2,
    fontsize=10
)
plt.title("Visualización del Árbol de Decisión Entrenado (max_depth=3)", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Reglas exactas en formato texto con conteo de muestras por clase [No Compra, Compra]
print("Reglas exactas del árbol entrenado (conteo en train: [No Compra, Compra]):")
print(export_text(dt_interpretable, feature_names=list(X_train_proc.columns), show_weights=True))


Reglas exactas del árbol entrenado (conteo en train: [No Compra, Compra]):
|--- PageValues <= 1.11
|   |--- Month_Nov <= 0.50
|   |   |--- Month_Sep <= 0.50
|   |   |   |--- weights: [5619.00, 87.00] class: 0
|   |   |--- Month_Sep >  0.50
|   |   |   |--- weights: [245.00, 28.00] class: 0
|   |--- Month_Nov >  0.50
|   |   |--- ProductRelated_Duration <= 1875.24
|   |   |   |--- weights: [1239.00, 95.00] class: 0
|   |   |--- ProductRelated_Duration >  1875.24
|   |   |   |--- weights: [331.00, 91.00] class: 0
|--- PageValues >  1.11
|   |--- BounceRates <= 0.00
|   |   |--- PageValues <= 23.61
|   |   |   |--- weights: [189.00, 260.00] class: 1
|   |   |--- PageValues >  23.61
|   |   |   |--- weights: [74.00, 466.00] class: 1
|   |--- BounceRates >  0.00
|   |   |--- Month_Nov <= 0.50
|   |   |   |--- weights: [490.00, 277.00] class: 0
|   |   |--- Month_Nov >  0.50
|   |   |   |--- weights: [151.00, 222.00] class: 1



### 4.2. Interpretación de reglas reales extraídas del árbol entrenado

Analizando directamente los nodos del árbol entrenado sobre las `9.864` sesiones de entrenamiento, extraemos **4 reglas de negocio reales y verificables**:

1. **Regla 1 — Descarte rápido por ausencia de valor de página (Rama izquierda principal):**
   > **SI** `PageValues <= 1.11` **Y** `Month_Nov <= 0.50` (no es noviembre) **Y** `Month_Sep <= 0.50` (no es septiembre) $\rightarrow$ **Predicción: `No Compra (0)`**.
   * **Evidencia en el árbol:** En esta hoja caen `5.706` sesiones de entrenamiento, de las cuales **`5.619` no compraron (98,48%)** y solo `87` compraron (`1,52%`, impureza `Gini = 0.03`).
   * **Interpretación de negocio:** Si el usuario navega fuera de noviembre/septiembre por páginas que históricamente no aportan valor transaccional (`PageValues <= 1.11`), la probabilidad de compra es prácticamente nula (1,5%).

2. **Regla 2 — Comprador de altísima probabilidad (Rama derecha sin rebote y alto `PageValues`):**
   > **SI** `PageValues > 1.11` **Y** `BounceRates <= 0.00` (tasa de rebote nula/casi nula) **Y** `PageValues > 23.61` $\rightarrow$ **Predicción: `Compra (1)`**.
   * **Evidencia en el árbol:** De las `540` sesiones que cumplen estas tres condiciones en entrenamiento, **`466` terminaron en compra (86,30%)** frente a solo `74` sin compra (`Gini = 0.24`).
   * **Interpretación de negocio:** Un usuario que interactúa sin rebotar (`BounceRates <= 0.00`) y llega a páginas de producto/checkout con alto valor transaccional (`PageValues > 23.61`) tiene un **86,3% de probabilidad de comprar**.

3. **Regla 3 — Comprador moderado sin rebote:**
   > **SI** `PageValues > 1.11` **Y** `BounceRates <= 0.00` **Y** `PageValues <= 23.61` (es decir, `1.11 < PageValues <= 23.61`) $\rightarrow$ **Predicción: `Compra (1)`**.
   * **Evidencia en el árbol:** Agrupa `449` sesiones en entrenamiento, con **`260` compras (57,91%)** y `189` no compras (`Gini = 0.49`).

4. **Regla 4 — El efecto *Black Friday* / Noviembre compensa tasas de rebote mayores:**
   > **SI** `PageValues > 1.11` **Y** `BounceRates > 0.00` **Y** `Month_Nov > 0.50` (la sesión ocurre en **noviembre**) $\rightarrow$ **Predicción: `Compra (1)`** (`222` compras vs `151` no compras, **59,52% de conversión**).
   > **PERO SI** la misma sesión con `BounceRates > 0.00` ocurre en **cualquier otro mes** (`Month_Nov <= 0.50`) $\rightarrow$ **Predicción: `No Compra (0)`** (`490` no compras vs `277` compras, solo **36,11% de conversión**).
   * **Interpretación de negocio:** Cuando el usuario visita páginas con valor (`PageValues > 1.11`) pero con cierta tasa de rebote (`BounceRates > 0`), el modelo predice compra únicamente si estamos en campaña de **noviembre**.


---
## 5. Control de complejidad y *overfitting*

Un árbol de decisión sin restricciones sigue dividiendo el espacio hasta que cada hoja es pura (`Gini = 0.0`), memorizando el conjunto de entrenamiento. Vamos a estudiar sistemáticamente cómo afectan los tres hiperparámetros clave de regularización (*pre-pruning*):
* **`max_depth`**: Profundidad máxima del árbol (número máximo de preguntas encadenadas desde la raíz).
* **`min_samples_split`**: Número mínimo de muestras requeridas en un nodo interno para permitir dividirlo.
* **`min_samples_leaf`**: Número mínimo de muestras que debe contener obligatoriamente cada nodo hoja final.


In [12]:
# Experimento sistemático variando max_depth, min_samples_split y min_samples_leaf
depths = list(range(1, 21))
splits = [2, 5, 10, 25, 50, 100, 150, 200, 300, 500]
leaves = [1, 2, 5, 10, 20, 35, 50, 75, 100, 150, 250]

def evaluar_curva_hiperparametro(param_name, valores):
    resultados = []
    for val in valores:
        kwargs = {param_name: val, 'random_state': RANDOM_STATE}
        dt = DecisionTreeClassifier(**kwargs)
        dt.fit(X_train_proc, y_train)
        resultados.append({
            'valor': val,
            'Acc_Train': accuracy_score(y_train, dt.predict(X_train_proc)),
            'Acc_Test': accuracy_score(y_test, dt.predict(X_test_proc)),
            'F1_Train': f1_score(y_train, dt.predict(X_train_proc)),
            'F1_Test': f1_score(y_test, dt.predict(X_test_proc)),
            'N_Hojas': dt.get_n_leaves()
        })
    return pd.DataFrame(resultados)

df_depth = evaluar_curva_hiperparametro('max_depth', depths)
df_split = evaluar_curva_hiperparametro('min_samples_split', splits)
df_leaf = evaluar_curva_hiperparametro('min_samples_leaf', leaves)

# Gráficas comparativas de Entrenamiento vs Test (F1-Score y Accuracy)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Curva max_depth
axes[0].plot(df_depth['valor'], df_depth['F1_Train'], 'o-', label='F1 Train', color='#1f77b4', linewidth=2)
axes[0].plot(df_depth['valor'], df_depth['F1_Test'], 's-', label='F1 Test', color='#ff7f0e', linewidth=2)
axes[0].plot(df_depth['valor'], df_depth['Acc_Train'], '--', label='Acc Train', color='#1f77b4', alpha=0.5)
axes[0].plot(df_depth['valor'], df_depth['Acc_Test'], '--', label='Acc Test', color='#ff7f0e', alpha=0.5)
axes[0].axvline(x=4, color='green', linestyle=':', label='Zona óptima (depth=4-6)')
axes[0].set_title('Efecto de max_depth (Train vs Test)', fontweight='bold')
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('Puntuación (F1 / Accuracy)')
axes[0].legend()

# 2. Curva min_samples_split
axes[1].plot(df_split['valor'], df_split['F1_Train'], 'o-', label='F1 Train', color='#1f77b4', linewidth=2)
axes[1].plot(df_split['valor'], df_split['F1_Test'], 's-', label='F1 Test', color='#ff7f0e', linewidth=2)
axes[1].set_title('Efecto de min_samples_split (con max_depth=None)', fontweight='bold')
axes[1].set_xlabel('min_samples_split')
axes[1].set_ylabel('F1-Score')
axes[1].legend()

# 3. Curva min_samples_leaf
axes[2].plot(df_leaf['valor'], df_leaf['F1_Train'], 'o-', label='F1 Train', color='#1f77b4', linewidth=2)
axes[2].plot(df_leaf['valor'], df_leaf['F1_Test'], 's-', label='F1 Test', color='#ff7f0e', linewidth=2)
axes[2].set_title('Efecto de min_samples_leaf (con max_depth=None)', fontweight='bold')
axes[2].set_xlabel('min_samples_leaf')
axes[2].set_ylabel('F1-Score')
axes[2].legend()

plt.suptitle('Control de complejidad y Overfitting en Decision Tree', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

display(df_depth.set_index('valor').loc[[1, 2, 3, 4, 5, 6, 8, 10, 15, 20]])


,Acc_Train,Acc_Test,F1_Train,F1_Test,N_Hojas
valor,,,,,
1,0.8778,0.8686,0.6703,0.6432,2
2,0.8922,0.8861,0.5773,0.5293,4
3,0.8994,0.8970,0.6565,0.6392,8
4,0.9046,0.8962,0.6252,0.5817,16
5,0.9102,0.8994,0.6808,0.6457,32
6,0.9171,0.8942,0.7079,0.6201,60
8,0.9312,0.8921,0.7532,0.6134,165
10,0.9498,0.8824,0.8265,0.5869,313
15,0.9886,0.8654,0.9628,0.5526,674


In [13]:
# Búsqueda de la mejor combinación de hiperparámetros para Decision Tree mediante validación cruzada
param_grid_dt = {
    'max_depth': [4, 5, 6],
    'min_samples_split': [25, 50],
    'min_samples_leaf': [15, 25],
    'class_weight': [None, 'balanced']
}

grid_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid_dt,
    scoring='f1',
    cv=5,
    n_jobs=1
)
grid_dt.fit(X_train_proc, y_train)

dt_optimizado = grid_dt.best_estimator_
print(f"Mejores hiperparámetros encontrados (CV=5 optimizando F1): {grid_dt.best_params_}")
print(f"Número de hojas del árbol optimizado: {dt_optimizado.get_n_leaves()} (frente a {dt_sin_limite.get_n_leaves()} del árbol sin límite)")

res_dt_opt = evaluar_modelo("Decision Tree Optimizado (Podado + Balanced)", dt_optimizado, X_train_proc, y_train, X_test_proc, y_test)

# También guardamos un árbol podado sin class_weight para comparar el efecto puro de la poda
dt_podado_unweighted = DecisionTreeClassifier(max_depth=6, min_samples_leaf=25, min_samples_split=50, random_state=RANDOM_STATE)
dt_podado_unweighted.fit(X_train_proc, y_train)
res_dt_podado = evaluar_modelo("Decision Tree Podado (max_depth=6, leaf=25, sin balanceo)", dt_podado_unweighted, X_train_proc, y_train, X_test_proc, y_test)


Mejores hiperparámetros encontrados (CV=5 optimizando F1): {'class_weight': None, 'max_depth': 4, 'min_samples_leaf': 15, 'min_samples_split': 25}
Número de hojas del árbol optimizado: 16 (frente a 815 del árbol sin límite)
📊 Resultados de: Decision Tree Optimizado (Podado + Balanced)
   • Entrenamiento -> Accuracy: 0.9046 | F1-Score: 0.6252
   • Test          -> Accuracy: 0.8962 | Precision: 0.7739 | Recall: 0.4660 | F1-Score: 0.5817 | ROC-AUC: 0.9093

📊 Resultados de: Decision Tree Podado (max_depth=6, leaf=25, sin balanceo)
   • Entrenamiento -> Accuracy: 0.9130 | F1-Score: 0.6866
   • Test          -> Accuracy: 0.8998 | Precision: 0.7288 | Recall: 0.5628 | F1-Score: 0.6352 | ROC-AUC: 0.9171



### ❓ Pregunta de análisis (Sección 5)
> **¿Qué señales indican que un árbol está sobreajustando (*overfitting*)?**

**Respuesta razonada:**
A partir de las curvas y tablas obtenidas, identificamos **cuatro señales inequívocas de sobreajuste**:
1. **Brecha creciente (*gap*) entre el rendimiento en entrenamiento y en test:**
   * A partir de `max_depth = 6`, el **F1-score en entrenamiento se dispara desde `0.7036` hasta `0.9964`** (y `1.0000` en profundidad 24), mientras que el **F1-score en test cae desde `0.6350` hasta `0.5438`** (y el ROC-AUC cae drásticamente de `0.91` a `0.73`).
2. **Rendimiento casi perfecto en entrenamiento (`Acc_Train ≈ 1.00`, `F1_Train ≈ 1.00`):**
   * En un problema real con comportamiento humano ruidoso (usuarios que abandonan el carrito por motivos externos), alcanzar un 100% de exactitud en entrenamiento significa que el modelo ha memorizado el ruido y las particularidades de cada individuo en lugar de aprender patrones generales.
3. **Explosión en el número de nodos hoja y profundidad excesiva:**
   * El árbol sin restricciones alcanza **24 niveles de profundidad y 815 hojas** (frente a solo 21-38 hojas en los árboles regularizados), creando reglas hiperespecíficas del tipo *"si `ProductRelated_Duration` está entre 1.412,3 y 1.415,1 segundos..."*.
4. **Hojas con muy pocas muestras (`min_samples_leaf = 1`):**
   * Cuando cuando `min_samples_leaf` o `min_samples_split` son muy pequeños (`1` o `2`), el árbol aísla casos atípicos individuales en hojas de 1 sola muestra, disparando la varianza del modelo ante datos nuevos. Al exigir `min_samples_leaf >= 25` o limitar `max_depth` entre `4` y `6`, las curvas de entrenamiento y test convergen y el modelo generaliza correctamente.


---
## 6. Regiones de decisión en 2D

Para comprender geométricamente cómo divide el espacio un árbol de decisión, seleccionamos las **dos variables continuas con mayor poder discriminativo y sentido de negocio**:
1. **`PageValues` (Eje X):** Valor económico medio de las páginas visitadas.
2. **`BounceRates` (Eje Y):** Tasa media de rebote de las páginas visitadas.

Entrenamos un árbol bidimensional (`max_depth=3`, `min_samples_leaf=50`) empleando **únicamente estas dos variables** y representamos lado a lado:
* Las **regiones de decisión 2D** con las fronteras ortogonales exactas trazadas por el modelo.
* El **árbol de decisión 2D** con sus condiciones `if/then` en cada nodo.


In [14]:
# Selección de las 2 variables para el estudio bidimensional
features_2d = ['PageValues', 'BounceRates']
X_train_2d = X_train_proc[features_2d].copy()
X_test_2d = X_test_proc[features_2d].copy()

# Entrenamos el árbol 2D con profundidad 3
dt_2d = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=RANDOM_STATE)
dt_2d.fit(X_train_2d, y_train)

print(f"Rendimiento del Árbol 2D (solo PageValues y BounceRates) en Test:")
print(f"Accuracy: {accuracy_score(y_test, dt_2d.predict(X_test_2d)):.4f} | F1-Score: {f1_score(y_test, dt_2d.predict(X_test_2d)):.4f}")
print("\nReglas aprendidas por el Árbol 2D:")
print(export_text(dt_2d, feature_names=features_2d, show_weights=True))


Rendimiento del Árbol 2D (solo PageValues y BounceRates) en Test:
Accuracy: 0.8832 | F1-Score: 0.6260

Reglas aprendidas por el Árbol 2D:
|--- PageValues <= 1.11
|   |--- PageValues <= 0.05
|   |   |--- BounceRates <= 0.02
|   |   |   |--- weights: [5441.00, 262.00] class: 0
|   |   |--- BounceRates >  0.02
|   |   |   |--- weights: [1945.00, 24.00] class: 0
|   |--- PageValues >  0.05
|   |   |--- weights: [48.00, 15.00] class: 0
|--- PageValues >  1.11
|   |--- BounceRates <= 0.00
|   |   |--- PageValues <= 23.61
|   |   |   |--- weights: [189.00, 260.00] class: 1
|   |   |--- PageValues >  23.61
|   |   |   |--- weights: [74.00, 466.00] class: 1
|   |--- BounceRates >  0.00
|   |   |--- PageValues <= 10.53
|   |   |   |--- weights: [340.00, 188.00] class: 0
|   |   |--- PageValues >  10.53
|   |   |   |--- weights: [301.00, 311.00] class: 1



In [15]:
# Representación bidimensional de las Regiones de Decisión + Árbol 2D lado a lado
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))

# Malla 2D acotada al rango donde se concentra el 98% de los datos (PageValues: 0 a 60, BounceRates: 0 a 0.10)
x_min, x_max = -1.0, 60.0
y_min, y_max = -0.003, 0.10
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 500), np.linspace(y_min, y_max, 500))
grid_points = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=features_2d)
Z_class = dt_2d.predict(grid_points).reshape(xx.shape)

# 1. Gráfico de regiones de decisión
axes[0].contourf(xx, yy, Z_class, alpha=0.30, levels=[-0.5, 0.5, 1.5], colors=['#4C72B0', '#55A868'])
axes[0].contour(xx, yy, Z_class, levels=[0.5], colors='black', linewidths=2.2)

# Líneas de corte de los nodos internos del árbol
axes[0].axvline(x=1.11, color='#D62728', linestyle='--', linewidth=2, label='Nodo Raíz: PageValues = 1.11')
axes[0].axhline(y=0.0002, xmin=(1.11 - x_min)/(x_max - x_min), xmax=1.0, color='#9467BD', linestyle='-.', linewidth=2, label='Nivel 1 (Der): BounceRates ≈ 0.00')
axes[0].axvline(x=10.53, ymin=(0.0002 - y_min)/(y_max - y_min), ymax=1.0, color='#FF7F0E', linestyle=':', linewidth=2.2, label='Nivel 2: PageValues = 10.53')

# Muestra de puntos reales del conjunto de test
muestra_test = X_test_2d.copy()
muestra_test['Revenue'] = y_test
muestra_test = muestra_test[(muestra_test['PageValues'] <= 60) & (muestra_test['BounceRates'] <= 0.10)].sample(800, random_state=RANDOM_STATE)

sns.scatterplot(
    data=muestra_test, x='PageValues', y='BounceRates', hue='Revenue',
    palette={0: '#1f4e78', 1: '#2ca02c'}, style='Revenue', s=40, alpha=0.75, ax=axes[0]
)

# Etiquetas de las regiones en el plano
axes[0].text(0.1, 0.085, "Región R1\nPageValues <= 1.11\n(No Compra)", fontsize=9.5, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))
axes[0].text(4.8, 0.055, "Región R2\n1.11 < PV <= 10.53\nBounceRates > 0\n(No Compra)", fontsize=9.5, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))
axes[0].text(28.0, 0.055, "Región R3\nPageValues > 10.53\nBounceRates > 0\n(COMPRA)", fontsize=9.5, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#d9f2d9', alpha=0.9))
axes[0].text(28.0, 0.008, "Región R4: PageValues > 1.11 y BounceRates ≈ 0 (COMPRA)", fontsize=9.5, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#d9f2d9', alpha=0.9))

axes[0].set_xlim(x_min, x_max)
axes[0].set_ylim(y_min, y_max)
axes[0].set_title('Regiones de Decisión 2D (PageValues vs BounceRates)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('PageValues (Valor medio de página)')
axes[0].set_ylabel('BounceRates (Tasa de rebote)')
axes[0].legend(loc='upper right', fontsize=8.5)

# 2. Visualización del árbol 2D
plot_tree(
    dt_2d,
    feature_names=features_2d,
    class_names=['No Compra (0)', 'Compra (1)'],
    filled=True,
    rounded=True,
    precision=2,
    fontsize=9,
    ax=axes[1]
)
axes[1].set_title('Estructura del Árbol 2D que genera las particiones', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


### 🔗 Explicación de la relación entre el espacio 2D, las condiciones `if/then` y las ramas del árbol

1. **Las divisiones del espacio son hiperplanos ortogonales (rectas paralelas a los ejes):**
   * Como cada nodo interno del árbol evalúa **una única variable a la vez** comparándola con un umbral constante ($X_j \le t$), geométricamente cada condición crea un corte perpendicular al eje de esa variable:
     * Una pregunta sobre `PageValues` genera una **recta vertical** ($x = t$).
     * Una pregunta sobre `BounceRates` genera una **recta horizontal** ($y = t$).
   * En consecuencia, un árbol de decisión divide el plano bidimensional en **regiones rectangulares disjuntas** (*axis-aligned rectangles*).

2. **Correspondencia exacta con las condiciones `if/then` y las ramas del modelo entrenado:**
   * **Nodo Raíz (`if PageValues <= 1.11`):** Trazado como la línea vertical roja discontinua en $x = 1.11$. Divide todo el plano en dos semiplanos:
     * **Rama izquierda (`PageValues <= 1.11` $\rightarrow$ Región R1):** Aunque el árbol hace sub-cortes internos en `PageValues <= 0.05` y `BounceRates <= 0.02` para refinar las probabilidades, todas las hojas de esta rama izquierda predicen **`No Compra (0)`** (zona azul izquierda).
     * **Rama derecha (`PageValues > 1.11`):** En el semiplano derecho ($x > 1.11$), el siguiente nodo evalúa **`if BounceRates <= 0.00`**:
       * **Sub-rama inferior (`BounceRates == 0` $\rightarrow$ Región R4):** Corresponde a la franja horizontal pegada al eje inferior ($y = 0, x > 1.11$), donde las hojas predicen **`Compra (1)`** (zona verde inferior).
       * **Sub-rama superior (`BounceRates > 0`):** Aquí el nodo de nivel 2 aplica un nuevo corte vertical **`if PageValues <= 10.53`** (línea naranja punteada en $x = 10.53$), separando el rectángulo central **Región R2 (`No Compra`, $1.11 < x \le 10.53$)** del rectángulo derecho **Región R3 (`Compra`, $x > 10.53$)**.


---
## 7. Random Forest

Entrenamos ahora un modelo ensamblado **Random Forest (*Bosque Aleatorio*)** utilizando **exactamente las mismas variables (`X_train_proc`, `X_test_proc`) y la misma partición estratificada** empleada para el Decision Tree.

### Justificación de hiperparámetros del Random Forest
1. **`n_estimators = 200` (Número de árboles):**
   * Un Random Forest combina las predicciones de $B$ árboles entrenados sobre muestras *bootstrap* independientes y seleccionando en cada corte un subconjunto aleatorio de variables (`max_features='sqrt'`, es decir, $\sqrt{28} \approx 5$ variables por nodo).
   * Como comprobamos en el gráfico inferior, a partir de `150 - 200` árboles la varianza del ensamblado se estabiliza por completo sin aumentar innecesariamente el tiempo de cómputo.
2. **`max_depth = 12` y `min_samples_leaf = 5` (Limitación de profundidad):**
   * Comparamos un Random Forest **sin límite de profundidad (`max_depth=None`)** frente a un Random Forest **con profundidad controlada (`max_depth=12, min_samples_leaf=5`)** y su variante con **`class_weight='balanced_subsample'`**. Limitar la profundidad máxima a `12` evita que cada árbol individual memorice ruido extremo, reduciendo el sobreajuste en entrenamiento (de `1.0000` a `0.91-0.94`) y mejorando el `F1-Score` y `ROC-AUC` en test.
3. **`random_state = 42`:**
   * Fija la semilla aleatoria tanto para el muestreo *bootstrap* de las filas de cada árbol como para el muestreo aleatorio de columnas en cada división, garantizando reproducibilidad exacta.


In [16]:
# Estudio del número de árboles (n_estimators) para justificar la elección de n_estimators=200
lista_n_arboles = [10, 25, 50, 100, 150, 200, 300]
f1_por_n = []
auc_por_n = []

for n in lista_n_arboles:
    rf_temp = RandomForestClassifier(n_estimators=n, max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=1)
    rf_temp.fit(X_train_proc, y_train)
    f1_por_n.append(f1_score(y_test, rf_temp.predict(X_test_proc)))
    auc_por_n.append(roc_auc_score(y_test, rf_temp.predict_proba(X_test_proc)[:, 1]))

plt.figure(figsize=(9, 4))
plt.plot(lista_n_arboles, f1_por_n, 'o-', label='F1-Score (Test)', color='#2ca02c', linewidth=2)
plt.plot(lista_n_arboles, auc_por_n, 's--', label='ROC-AUC (Test)', color='#1f77b4', linewidth=2)
plt.axvline(x=200, color='red', linestyle=':', label='Elegido: n_estimators=200')
plt.title('Estabilización del rendimiento de Random Forest según n_estimators', fontweight='bold')
plt.xlabel('Número de árboles (n_estimators)')
plt.ylabel('Métrica en Test')
plt.legend()
plt.tight_layout()
plt.show()


In [17]:
# Entrenamiento de las variantes de Random Forest sobre las mismas particiones
# 1. Random Forest Base (n_estimators=200, sin límite de profundidad)
rf_base = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=RANDOM_STATE,
    n_jobs=1
)
rf_base.fit(X_train_proc, y_train)
res_rf_base = evaluar_modelo("Random Forest Base (200 árboles, max_depth=None)", rf_base, X_train_proc, y_train, X_test_proc, y_test)

# 2. Random Forest Regularizado (n_estimators=200, max_depth=12, min_samples_leaf=5)
rf_reg = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=1
)
rf_reg.fit(X_train_proc, y_train)
res_rf_reg = evaluar_modelo("Random Forest Regularizado (200 árb., max_depth=12, leaf=5)", rf_reg, X_train_proc, y_train, X_test_proc, y_test)

# 3. Random Forest Optimizado para desequilibrio (n_estimators=200, max_depth=12, balanced_subsample)
rf_opt = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE,
    n_jobs=1
)
rf_opt.fit(X_train_proc, y_train)
res_rf_opt = evaluar_modelo("Random Forest Optimizado (200 árb., max_depth=12, balanced)", rf_opt, X_train_proc, y_train, X_test_proc, y_test)


📊 Resultados de: Random Forest Base (200 árboles, max_depth=None)
   • Entrenamiento -> Accuracy: 1.0000 | F1-Score: 1.0000
   • Test          -> Accuracy: 0.8982 | Precision: 0.7282 | Recall: 0.5471 | F1-Score: 0.6248 | ROC-AUC: 0.9211

📊 Resultados de: Random Forest Regularizado (200 árb., max_depth=12, leaf=5)
   • Entrenamiento -> Accuracy: 0.9398 | F1-Score: 0.7810
   • Test          -> Accuracy: 0.9027 | Precision: 0.7610 | Recall: 0.5419 | F1-Score: 0.6330 | ROC-AUC: 0.9235

📊 Resultados de: Random Forest Optimizado (200 árb., max_depth=12, balanced)
   • Entrenamiento -> Accuracy: 0.9118 | F1-Score: 0.7598
   • Test          -> Accuracy: 0.8715 | Precision: 0.5639 | Recall: 0.7513 | F1-Score: 0.6442 | ROC-AUC: 0.9260



---
## 8. Evaluación y tabla comparativa de modelos

Reunimos en una **tabla comparativa global** todas las métricas requeridas (`Accuracy`, `Precision`, `Recall`, `F1-Score` y `ROC-AUC`, tanto en entrenamiento como en test) y representamos las **matrices de confusión** para comparar el tipo de aciertos y errores de cada estrategia.


In [18]:
# Tabla comparativa global de todos los modelos entrenados
tabla_comparativa = pd.DataFrame([
    res_dt_sin_limite,
    res_dt_interpretable,
    res_dt_podado,
    res_dt_opt,
    res_rf_base,
    res_rf_reg,
    res_rf_opt
]).set_index('Modelo')

display(tabla_comparativa.style.background_gradient(cmap='Greens', subset=['Accuracy (Test)', 'Precision (Test)', 'Recall (Test)', 'F1-Score (Test)', 'ROC-AUC (Test)']).format("{:.4f}"))


,Acc Train,F1 Train,Accuracy (Test),Precision (Test),Recall (Test),F1-Score (Test),ROC-AUC (Test)
Modelo,,,,,,,
"Decision Tree (Sin límite, max_depth=None)",1.0000,1.0000,0.8564,0.5355,0.5524,0.5438,0.7323
Decision Tree Interpretable (max_depth=3),0.8994,0.6565,0.8970,0.6988,0.5890,0.6392,0.9011
"Decision Tree Podado (max_depth=6, leaf=25, sin balanceo)",0.9130,0.6866,0.8998,0.7288,0.5628,0.6352,0.9171
Decision Tree Optimizado (Podado + Balanced),0.9046,0.6252,0.8962,0.7739,0.4660,0.5817,0.9093
"Random Forest Base (200 árboles, max_depth=None)",1.0000,1.0000,0.8982,0.7282,0.5471,0.6248,0.9211
"Random Forest Regularizado (200 árb., max_depth=12, leaf=5)",0.9398,0.7810,0.9027,0.7610,0.5419,0.6330,0.9235
"Random Forest Optimizado (200 árb., max_depth=12, balanced)",0.9118,0.7598,0.8715,0.5639,0.7513,0.6442,0.9260


In [19]:
# Visualización comparativa de las Matrices de Confusión en el conjunto de Test (n = 2.466)
modelos_cm = [
    ("DT Sin Límite\n(Overfitting)", dt_sin_limite),
    ("DT Optimizado\n(Podado + Balanced)", dt_optimizado),
    ("Random Forest Reg.\n(Sin balanceo)", rf_reg),
    ("Random Forest Opt.\n(Con balanceo)", rf_opt)
]

fig, axes = plt.subplots(1, 4, figsize=(20, 4.6))

for ax, (titulo, mod) in zip(axes, modelos_cm):
    cm = confusion_matrix(y_test, mod.predict(X_test_proc))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Compra (0)', 'Compra (1)'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.grid(False)

plt.suptitle('Comparativa de Matrices de Confusión en Test (2.466 sesiones: 2.084 No Compra, 382 Compra)', fontsize=13, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()


### 📊 Análisis detallado de las métricas y matrices de confusión

1. **Superación del *overfitting* del árbol sin límite:**
   * El `Decision Tree (Sin límite)` memoriza el 100% del entrenamiento (`F1 Train = 1.0000`), pero en test es el peor modelo de todos (`F1 Test = 0.5438`, `ROC-AUC = 0.7323`), cometiendo `183` falsos positivos y `171` falsos negativos.
   * Simplemente podando el árbol (`max_depth=6, min_samples_leaf=25`), el `F1-Score` en test sube de **`0.5438` a `0.6254`** y el `ROC-AUC` da un salto de **`0.7323` a `0.9163`**.

2. **Superioridad del Random Forest frente al árbol individual:**
   * A igualdad de preprocesamiento y sin balanceo, el **`Random Forest Regularizado`** alcanza la **mayor Accuracy global en test (`90.27%`)**, con una **Precision del `76.10%`**, `F1-Score = 0.6330` y `ROC-AUC = 0.9235`, reduciendo los falsos positivos a solo `65` sesiones.
   * Si el objetivo comercial es **maximizar la detección de compradores reales (Recall)** y el **F1-Score global**, el **`Random Forest Optimizado (max_depth=12, balanced_subsample)`** obtiene el **mejor F1-Score de toda la práctica (`0.6442`)** y el **mayor ROC-AUC (`0.9260`)**, detectando **`287` de las `382` compras reales (`Recall = 75.13%`)** frente a las `207-211` (`~54%`) de los modelos sin balanceo de clases.


---
## 9. Importancia de variables (*Feature Importance*)

Extraemos la importancia de las variables (`feature_importances_`, basada en la reducción media de impureza de Gini ponderada en los 200 árboles del Random Forest) y la comparamos gráficamente.


In [20]:
# Obtención y representación gráfica de la importancia de variables del Random Forest
importancias_rf = pd.Series(rf_reg.feature_importances_, index=X_train_proc.columns).sort_values(ascending=False)
importancias_rf_opt = pd.Series(rf_opt.feature_importances_, index=X_train_proc.columns).sort_values(ascending=False)

tabla_imp = pd.DataFrame({
    'Importancia RF Regularizado (%)': importancias_rf * 100,
    'Importancia RF Balanced (%)': importancias_rf_opt.reindex(importancias_rf.index) * 100,
    'Importancia Acumulada (%)': importancias_rf.cumsum() * 100
})
display(tabla_imp.head(15))

plt.figure(figsize=(11, 6.5))
top_n = 15
sns.barplot(
    x=importancias_rf.head(top_n).values * 100,
    y=importancias_rf.head(top_n).index,
    palette='viridis',
    edgecolor='black'
)
for i, v in enumerate(importancias_rf.head(top_n).values * 100):
    plt.text(v + 0.4, i, f"{v:.2f}%", va='center', fontsize=9.5, fontweight='bold')

plt.title('Top 15 Variables más Importantes en Random Forest (% Reducción de Impureza de Gini)', fontsize=13, fontweight='bold')
plt.xlabel('Importancia relativa (%)')
plt.ylabel('Variable predictora')
plt.xlim(0, 50)
plt.tight_layout()
plt.show()


,Importancia RF Regularizado (%),Importancia RF Balanced (%),Importancia Acumulada (%)
PageValues,53.9915,49.4309,53.9915
ExitRates,7.5241,9.1562,61.5156
ProductRelated_Duration,7.0168,8.3135,68.5324
ProductRelated,5.1833,4.8584,73.7157
BounceRates,4.5931,5.0270,78.3088
Administrative_Duration,3.9856,3.9108,82.2944
Administrative,3.1219,2.7654,85.4163
Month_Nov,2.6388,3.2325,88.0551
Informational_Duration,1.6480,1.3449,89.7031
TrafficType,1.5670,1.5789,91.2701


### ❓ Respuestas a las preguntas de análisis (Sección 9)

#### 1. ¿Qué variables han sido más importantes?
Las **6 variables más determinantes** acumulan por sí solas más del **81% de la importancia total** del modelo:
1. **`PageValues` (`53.99%` en RF Regularizado / `49.43%` en RF Balanced):** Es, con enorme diferencia, la variable reina del modelo, aportando cerca de la mitad de la capacidad predictiva total.
2. **`ExitRates` (`7.52%` - `9.16%`):** La tasa media de salida de las páginas visitadas.
3. **`ProductRelated_Duration` (`7.02%` - `8.31%`):** El tiempo total invertido navegando en páginas de productos.
4. **`ProductRelated` (`5.18%`):** La cantidad de páginas de producto visitadas.
5. **`BounceRates` (`4.59%` - `5.03%`):** La tasa media de rebote.
6. **`Administrative_Duration` (`3.99%`)** y **`Administrative` (`3.12%`)**: La gestión de cuenta de usuario (login, dirección de envío, perfil).
7. Entre las variables categóricas, destaca **`Month_Nov` (`2.64%` - `3.23%`)** por el efecto campaña de noviembre (*Black Friday*).

#### 2. ¿Coincide con lo que esperabas al explorar los datos?
**Sí, coincide plenamente con lo observado en el análisis exploratorio (Sección 2):**
* En el EDA vimos que `PageValues` tenía la mayor correlación con `Revenue` (`r = 0.49`) y una diferencia abismal de medias (`27.26` en compradores frente a `1.98` en no compradores).
* También detectamos que los compradores dedicaban casi el doble de tiempo a páginas de producto (`1.876 s` vs `1.070 s`), visitaban más páginas administrativas (`3.39` vs `2.12`, necesario para iniciar sesión o gestionar datos de envío) y presentaban tasas de salida (`ExitRates`) y rebote (`BounceRates`) mucho más bajas.
* En cambio, variables de entorno técnico como `Browser`, `OperatingSystems` o `Weekend` mostraban distribuciones casi idénticas entre compradores y no compradores en el EDA, y efectivamente reciben una importancia residual (`< 1.5%`) en el Random Forest.

#### 3. ¿Una importancia alta implica causalidad? Justifica la respuesta.
**Rotundamente NO: una alta importancia predictiva mide asociación estadística (reducción de impureza), pero jamás implica relación causal directa.**
* **Justificación técnica y de negocio:**
  1. **Naturaleza de `PageValues` (Correlación por construcción / causa inversa):** En Google Analytics, `PageValues` asigna un valor alto a las páginas que están muy cerca del paso final del embudo de pago (por ejemplo, la página de *"Resumen del carrito"* o *"Seleccionar método de pago"*). No es que mostrar esa página a un usuario al azar le "cause" ganas de comprar; al contrario, **es la intención previa de comprar del usuario (variable latente o de confusión) la que causa que el usuario navegue hasta las páginas de checkout con alto `PageValues` y finalmente compre**.
  2. **Intervención contrafactual:** Si la tienda online forzara artificialmente a todos los visitantes a aterrizar directamente en una página con alto `PageValues` o les impidiera cerrar la pestaña para aumentar `ProductRelated_Duration`, **no aumentaría las ventas** (de hecho, probablemente frustraría al usuario).
  3. **Para demostrar causalidad** no basta con `feature_importances_` sobre datos observacionales; sería necesario realizar **experimentos controlados aleatorizados (A/B testing)** o inferencia causal.


---
## 10. Comparación final: Decision Tree vs. Random Forest

Comparamos de forma crítica ambas familias de modelos atendiendo a los 6 criterios técnicos y de negocio:

| Criterio | Decision Tree (Árbol Individual) | Random Forest (Bosque Aleatorio) |
| :--- | :--- | :--- |
| **1. Rendimiento predictivo** | **Moderado-Alto si se poda correctamente** (`F1 = 0.625 - 0.639`, `ROC-AUC = 0.917`). Inferior si no se poda (`F1 = 0.544`, `ROC-AUC = 0.732`). | **Superior en todas las métricas** (`Accuracy = 90.27%` en versión regularizada; `F1 = 0.6442` y `ROC-AUC = 0.9260` en versión balanceada). |
| **2. Interpretabilidad global** | **Muy alta (Caja Blanca)** cuando se limita la profundidad (`max_depth <= 4`). Se puede dibujar el diagrama completo del árbol y validar sus reglas de un vistazo. | **Baja (Caja Negra / Gris).** Al promediar 200 árboles distintos, no existe un único diagrama de reglas equivalente; solo podemos inspeccionar la importancia global de variables. |
| **3. Estabilidad** | **Baja (Alta varianza).** Pequeños cambios en el conjunto de entrenamiento (un 5% de sesiones distintas) pueden cambiar el nodo superior y alterar toda la estructura de ramas inferiores. | **Muy alta.** El promediado (*bagging*) de 200 árboles entrenados con muestreo *bootstrap* y selección aleatoria de variables suaviza las fronteras de decisión y lo hace robusto ante cambios en los datos. |
| **4. Riesgo de *overfitting*** | **Muy alto** por defecto. Exige un ajuste cuidadoso de `max_depth`, `min_samples_split` y `min_samples_leaf` para no memorizar el conjunto de entrenamiento. | **Mucho menor.** La combinación de *bagging* y submuestreo de variables (`max_features='sqrt'`) reduce drásticamente la varianza, y con un ligero control (`max_depth=12`) generaliza de forma excelente. |
| **5. Facilidad para explicar una predicción individual** | **Inmediata y transparente:** basta con recorrer el camino de la raíz a la hoja (3 a 5 condiciones `SI ... ENTONCES ...`) para justificar por qué una sesión fue clasificada como compra o no compra. | **Compleja:** requiere herramientas post-hoc externas (como valores *SHAP* o *LIME*) para desglosar la contribución de cada variable en los 200 árboles para una sesión concreta. |
| **6. Coste computacional** | **Mínimo (ultraligero):** entrena en milisegundos, ocupa pocos kilobytes en memoria y evalúa una sesión en apenas $O(\text{depth})$ comparaciones (`< 10` operaciones). | **Medio-Alto:** multiplica por $B=200$ el coste de almacenamiento e inferencia (requiere evaluar los 200 árboles por cada sesión en tiempo real, aunque es paralelizable). |

---
### 🎯 Justificación técnica: ¿Cuándo es preferible utilizar cada modelo?

* **¿Cuándo elegir el Decision Tree (podado)?**
  1. **Auditoría, explicabilidad normativa y comunicación con negocio:** Cuando el equipo de marketing o producto necesita **reglas explícitas de negocio** (por ejemplo, *"lanzar un cupón emergente si `PageValues > 1.11`, el usuario está en noviembre y su `ExitRate` supera 0.02"*) que puedan programarse directamente en el gestor de campañas web sin desplegar un servidor de Machine Learning.
  2. **Restricciones extremas de latencia o hardware:** Cuando la inferencia debe ejecutarse directamente en el navegador del cliente (JavaScript ligero) en microsegundos y sin coste de servidor.

* **¿Cuándo elegir el Random Forest?**
  1. **Sistemas de predicción automatizados en producción orientados a maximizar retorno (ROI):** Cuando el objetivo es lograr la **máxima precisión y discriminación (`ROC-AUC = 0.926`, `F1 = 0.644`)** en un motor de personalización en backend (por ejemplo, pujas automáticas de retargeting o priorización dinámica de ofertas), donde acertar más compras reales genera un beneficio económico directo muy superior al pequeño coste computacional de evaluar 200 árboles.
  2. **Entornos con datos ruidosos y cambiantes:** Donde la estabilidad del ensamblado evita que el modelo degrade su comportamiento ante fluctuaciones semanales del tráfico.


---
## 11. Análisis de errores

Seleccionamos **6 casos reales del conjunto de test mal clasificados** por nuestros modelos (3 **Falsos Positivos** y 3 **Falsos Negativos**) para inspeccionar sus valores de entrada, la clase real, la predicción emitida y razonar la causa del error desde la perspectiva del comportamiento en comercio electrónico.


In [21]:
# Construcción del DataFrame de errores sobre el conjunto de Test
df_errores = X_test.copy()
df_errores['Clase_Real (Revenue)'] = y_test
df_errores['Pred_RF_Opt'] = rf_opt.predict(X_test_proc)
df_errores['Prob_Compra_RF'] = np.round(rf_opt.predict_proba(X_test_proc)[:, 1], 3)
df_errores['Pred_DT_Opt'] = dt_optimizado.predict(X_test_proc)

# Seleccionamos 3 Falsos Positivos (Real = 0, Predicción = 1) y 3 Falsos Negativos (Real = 1, Predicción = 0)
falsos_positivos = df_errores[(df_errores['Clase_Real (Revenue)'] == 0) & (df_errores['Pred_RF_Opt'] == 1)].sort_values('Prob_Compra_RF', ascending=False)
falsos_negativos = df_errores[(df_errores['Clase_Real (Revenue)'] == 1) & (df_errores['Pred_RF_Opt'] == 0)].sort_values('Prob_Compra_RF', ascending=True)

casos_seleccionados = pd.concat([falsos_positivos.head(3), falsos_negativos.head(3)])
casos_seleccionados['Tipo_Error'] = ['Falso Positivo (FP)']*3 + ['Falso Negativo (FN)']*3

columnas_mostrar = [
    'Tipo_Error', 'Clase_Real (Revenue)', 'Pred_RF_Opt', 'Prob_Compra_RF', 'Pred_DT_Opt',
    'PageValues', 'ExitRates', 'BounceRates', 'ProductRelated', 'ProductRelated_Duration',
    'Administrative', 'Month', 'VisitorType', 'Weekend'
]

print("Tabla detallada de los 6 casos mal clasificados seleccionados del conjunto de Test:")
display(casos_seleccionados[columnas_mostrar])


Tabla detallada de los 6 casos mal clasificados seleccionados del conjunto de Test:


,Tipo_Error,Clase_Real (Revenue),Pred_RF_Opt,Prob_Compra_RF,Pred_DT_Opt,PageValues,ExitRates,BounceRates,ProductRelated,ProductRelated_Duration,Administrative,Month,VisitorType,Weekend
6429,Falso Positivo (FP),0,1,0.9270,1,166.3736,0.0063,0.0000,15,443.5667,2,June,New_Visitor,1
9486,Falso Positivo (FP),0,1,0.9060,1,15.9554,0.0087,0.0030,133,4885.0491,4,Nov,Returning_Visitor,1
11642,Falso Positivo (FP),0,1,0.9000,1,32.1392,0.0048,0.0000,101,2186.1458,7,Nov,Returning_Visitor,0
12086,Falso Negativo (FN),1,0,0.0620,0,0.0000,0.0667,0.0333,6,60.2500,0,Nov,Returning_Visitor,0
10789,Falso Negativo (FN),1,0,0.0690,0,0.0000,0.2000,0.2000,1,0.0000,1,Nov,Returning_Visitor,0
8958,Falso Negativo (FN),1,0,0.1060,0,0.0000,0.0517,0.0121,33,1412.3167,0,Nov,Returning_Visitor,0


### 🔍 Explicación individualizada de cada uno de los 6 casos mal clasificados

#### 🔴 Falsos Positivos (`Clase Real = 0 [No Compra]`, pero `Predicción = 1 [Compra]`): *Abandono de carrito en el último paso*

1. **Caso 1 — Sesión `#6429` (`Real = 0`, `Predicción = 1`, `Probabilidad RF = 0.927`):**
   * **Valores de entrada:** `PageValues = 166.37` (extremadamente alto), `BounceRates = 0.0000`, `ExitRates = 0.0063`, `ProductRelated = 15` páginas (`443.6 s`), `Administrative = 2`, `New_Visitor` en `June` (fin de semana).
   * **Explicación del error:** Esta sesión tiene absolutamente todas las señales estadísticas de una compra segura (`PageValues` altísimo de `166.37` y rebote nulo). Representa el clásico **abandono de carrito en la pasarela de pago final**: el usuario llegó hasta el último paso del checkout (disparando `PageValues`), pero canceló la compra en el último instante (por ejemplo, al ver los gastos de envío finales, por fallo en la tarjeta bancaria o por indecisión). Con las variables disponibles, es indistinguible de una sesión con compra.

2. **Caso 2 — Sesión `#9486` (`Real = 0`, `Predicción = 1`, `Probabilidad RF = 0.906`):**
   * **Valores de entrada:** `PageValues = 15.96`, `ProductRelated = 133` páginas, `ProductRelated_Duration = 4.885 s` (~81 minutos), `Administrative = 4`, `ExitRates = 0.0087`, `Month = Nov` (*Black Friday*), `Returning_Visitor`.
   * **Explicación del error:** Se trata de un usuario recurrente en pleno mes de **noviembre** que dedicó más de 1 hora y 20 minutos a comparar 133 páginas de productos e incluso entró en páginas con alto valor transaccional (`PageValues = 15.96`). Es un comportamiento típico de *window shopping* intensivo o preparación de cesta durante *Black Friday* sin llegar a confirmar el pedido en esa sesión concreta.

3. **Caso 3 — Sesión `#11642` (`Real = 0`, `Predicción = 1`, `Probabilidad RF = 0.900`):**
   * **Valores de entrada:** `PageValues = 32.14`, `BounceRates = 0.0000`, `ExitRates = 0.0048`, `ProductRelated = 101` páginas (`2.186 s`), `Administrative = 7`, `Month = Nov`, `Returning_Visitor`.
   * **Explicación del error:** De nuevo, un usuario que visitó 7 páginas administrativas (probablemente consultando su cuenta, direcciones o estado de pedidos), 101 páginas de producto sin ningún rebote y alcanzó un `PageValues = 32.14` en noviembre. Tanto el Decision Tree como el Random Forest predicen compra (`1`) porque más del 85% de las sesiones con este perfil compran, pero el usuario pospuso o abortó la transacción.

---

#### 🟠 Falsos Negativos (`Clase Real = 1 [Compra]`, pero `Predicción = 0 [No Compra]`): *Compras relámpago sin `PageValues` previo*

4. **Caso 4 — Sesión `#12086` (`Real = 1`, `Predicción = 0`, `Probabilidad RF = 0.062`):**
   * **Valores de entrada:** `PageValues = 0.00`, `ProductRelated = 6` páginas, `ProductRelated_Duration = 60.25 s` (solo 1 minuto), `Administrative = 0`, `BounceRates = 0.0333`, `ExitRates = 0.0667`, `Month = Nov`, `Returning_Visitor`.
   * **Explicación del error:** El usuario completó una compra en apenas **60 segundos** visitando solo 6 páginas y con **`PageValues = 0.00`**. Al tener `PageValues = 0` y una duración muy breve con tasa de salida alta (`0.0667`), ambos modelos lo clasifican como `No Compra`. Esto ocurre en **compras directas en 1 clic (*one-click buy*)** o cuando un visitante recurrente ya tenía el producto en su carrito de una sesión anterior y entró únicamente a finalizar el pago.

5. **Caso 5 — Sesión `#10789` (`Real = 1`, `Predicción = 0`, `Probabilidad RF = 0.069`):**
   * **Valores de entrada:** `PageValues = 0.00`, `ProductRelated = 1`, `ProductRelated_Duration = 0.0 s`, `Administrative = 1`, `BounceRates = 0.2000`, `ExitRates = 0.2000`, `Month = Nov`, `Returning_Visitor`.
   * **Explicación del error:** Es una anomalía o caso extremo del registro de sesiones: figura con solo 1 página administrativa y 1 de producto con duración `0.0` y tasa de rebote máxima (`0.20`), pero con `Revenue = True`. Corresponde a una recarga de página de confirmación o compra instantánea mediante enlace directo de pago donde el rastreador de analítica no acumuló `PageValues` previos.

6. **Caso 6 — Sesión `#8958` (`Real = 1`, `Predicción = 0`, `Probabilidad RF = 0.106`):**
   * **Valores de entrada:** `PageValues = 0.00`, `ProductRelated = 33` páginas (`1.412.3 s`, ~23 min), `Administrative = 0`, `BounceRates = 0.0121`, `ExitRates = 0.0517`, `Month = Nov`, `Returning_Visitor`.
   * **Explicación del error:** Aunque el usuario navegó durante 23 minutos por 33 páginas de producto en noviembre, la variable `PageValues` permaneció en `0.00` y no visitó ninguna página administrativa (`Administrative = 0`). Como los modelos otorgan más del 45% de importancia a `PageValues`, el valor `0.00` penaliza drásticamente la probabilidad estimada (`0.106`), generando un falso negativo.


---
## 12. Conclusiones finales y resumen de entrega

1. **Calidad y preprocesamiento del dataset:** El dataset *Online Shoppers Purchasing Intention* (12.330 sesiones, 0 nulos) presenta un notable desequilibrio de clases (`15,47%` compras frente a `84,53%` no compras), lo que invalida el uso aislado de la *Accuracy* y exige emplear particiones estratificadas, ajuste de transformaciones exclusivamente en entrenamiento (evitando *data leakage*) y evaluación mediante **Precision, Recall, F1-Score y ROC-AUC**.
2. **Decision Tree e interpretabilidad:** Un árbol de decisión podado (`max_depth=3` a `6`) permite extraer reglas de negocio directamente accionables y evita el severo *overfitting* del árbol sin restricciones (que pasaba de `F1 = 1.0000` en train a `0.5438` en test con 815 hojas).
3. **Random Forest y robustez:** Al combinar 200 árboles con `max_depth=12` y submuestreo aleatorio de variables, el **Random Forest supera al árbol individual en todas las métricas de generalización**, alcanzando un **90,27% de Accuracy** (`ROC-AUC = 0.9235`) en su versión regularizada y el **máximo F1-Score (`0.6442`) y Recall (`75,13%`, `ROC-AUC = 0.9260`)** al incorporar `class_weight='balanced_subsample'`.
4. **Factores clave y límites predictivos:** `PageValues`, `ExitRates`, `ProductRelated_Duration` y la estacionalidad (`Month_Nov`) concentran el grueso de la capacidad predictiva, aunque su importancia refleja asociación estadística y no causalidad directa. El análisis de errores demuestra que los fallos se concentran en **abandonos de carrito de última hora** (Falsos Positivos) y **compras directas sin rastro previo de `PageValues`** (Falsos Negativos).
